# 📈 Time Series Analysis — Memprediksi Masa Depan dari Data Waktu

**Modul 1: Machine Learning Fundamentals** | Notebook 9 dari 9 (Terakhir!)

---

## 🎯 Apa yang Akan Kita Pelajari?

1. Apa itu **Time Series** dan komponen-komponennya
2. Konsep **Trend, Seasonality, dan Noise**
3. Cara memeriksa **stationarity** (stasioneritas data)
4. Model **SARIMA** untuk forecasting
5. Mengevaluasi kualitas prediksi

### ⏱️ Estimasi Waktu: ~90 menit + diskusi

---

## 💡 Analogi Sederhana

Pernahkah kamu melihat **pola** di sekitarmu?

- 🌡️ Suhu udara naik di musim panas, turun di musim hujan → **berulang setiap tahun**
- 📱 Pengguna internet meningkat setiap tahun → **tren naik**
- 🏪 Penjualan es krim naik di bulan panas → **pola musiman**

**Time Series** = data yang berurutan berdasarkan waktu. Kita ingin menemukan **pola** di masa lalu untuk **memprediksi masa depan**.

### 3 Komponen Time Series:

| Komponen | Penjelasan | Contoh |
|----------|-----------|--------|
| **Trend** | Arah jangka panjang (naik/turun) | Populasi dunia terus naik |
| **Seasonality** | Pola yang berulang (musiman) | Penjualan payung naik di musim hujan |
| **Noise** | Variasi acak yang tidak bisa diprediksi | Fluktuasi harian yang random |

---

## 1️⃣ Persiapan

In [ ]:
!pip install -q pmdarima statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
import pmdarima as pm

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('✅ Library berhasil di-import!')

---

## 2️⃣ Memuat Data

### 📋 Tentang Dataset

Kita menggunakan dataset **SeaPlane Travel** — jumlah penumpang pesawat laut per bulan dari tahun 2003 sampai 2015 (144 bulan = 12 tahun).

**Pertanyaan:** Bisakah kita memprediksi **berapa penumpang** di bulan-bulan mendatang?

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Baca data dengan parsing tanggal
df = pd.read_csv('SeaPlaneTravel.csv', parse_dates=['Month'], index_col='Month')
df.columns = ['Passengers']

# Set frekuensi bulanan (penting untuk time series!)
df.index = pd.DatetimeIndex(df.index).to_period('M').to_timestamp()
df = df.asfreq('MS')  # Month Start frequency

# Cek apakah ada data kosong (gap dalam waktu)
missing = df['Passengers'].isnull().sum()
if missing > 0:
    print(f'⚠️ Ditemukan {missing} bulan data kosong (gap dalam dataset)')
    print(f'   Bulan yang kosong: {df[df["Passengers"].isnull()].index.strftime("%b %Y").tolist()}')
    # Interpolasi untuk mengisi gap
    df['Passengers'] = df['Passengers'].interpolate(method='linear')
    print(f'   ✅ Gap diisi dengan interpolasi linear\n')

print(f'Dataset: {len(df)} bulan data')
print(f'Periode: {df.index[0].strftime("%B %Y")} — {df.index[-1].strftime("%B %Y")}')
print(f'\nStatistik:')
print(f'  Minimum:  {df["Passengers"].min():.0f} penumpang')
print(f'  Rata-rata: {df["Passengers"].mean():.0f} penumpang')
print(f'  Maksimum: {df["Passengers"].max():.0f} penumpang')

df.head(10)

In [ ]:
# Visualisasi data
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Passengers'], color='#2196F3', linewidth=1.5)
plt.fill_between(df.index, df['Passengers'], alpha=0.1, color='#2196F3')
plt.xlabel('Tahun', fontsize=12)
plt.ylabel('Jumlah Penumpang', fontsize=12)
plt.title('Jumlah Penumpang Pesawat Laut per Bulan (2003-2015)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Apa yang kamu lihat?')
print('  1. TREND: Jumlah penumpang secara umum NAIK dari waktu ke waktu')
print('  2. SEASONALITY: Ada pola berulang setiap tahun (naik-turun musiman)')
print('  3. Amplitudo seasonality MEMBESAR seiring waktu (variasi makin besar)')

---

## 3️⃣ Dekomposisi — Memisahkan Komponen Time Series

Kita bisa **memecah** data time series menjadi 3 komponen terpisah: Trend, Seasonality, dan Residual (noise).

In [ ]:
# Dekomposisi time series
decomp = seasonal_decompose(df['Passengers'], model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

components = [
    ('Data Asli', df['Passengers'], '#2196F3'),
    ('Trend (Arah Jangka Panjang)', decomp.trend, '#4CAF50'),
    ('Seasonality (Pola Musiman)', decomp.seasonal, '#FF6F00'),
    ('Residual (Noise/Sisa)', decomp.resid, '#EF5350')
]

for ax, (title, data, color) in zip(axes, components):
    ax.plot(data, color=color, linewidth=1.5)
    ax.set_ylabel(title, fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('Dekomposisi Time Series', fontsize=15, fontweight='bold', y=1.01)
plt.xlabel('Tahun', fontsize=11)
plt.tight_layout()
plt.show()

print('📊 Penjelasan:')
print('  🔵 Data Asli = Trend × Seasonality × Residual (model multiplikatif)')
print('  🟢 Trend: penumpang terus meningkat dari ~100 ke ~500')
print('  🟠 Seasonality: pola berulang 12 bulan (puncak di bulan tertentu)')
print('  🔴 Residual: variasi acak yang tidak mengikuti pola')

---

## 4️⃣ Stationarity — Apakah Data "Stabil"?

Kebanyakan model time series membutuhkan data yang **stasioner** — artinya rata-rata dan variasi data **tidak berubah** seiring waktu.

Data kita jelas **tidak stasioner** karena ada trend naik! Kita perlu "menstabilkan" data dengan **differencing** (menghitung selisih antar periode).

**Analogi:** Bayangkan kamu mencatat tinggi badanmu setiap tahun:
- Data asli: 150, 155, 159, 162 → NAIK terus (tidak stasioner)
- Selisih: +5, +4, +3 → Lebih stabil! (mendekati stasioner)

### Augmented Dickey-Fuller (ADF) Test
Tes statistik untuk mengecek stasioneritas:
- **p-value < 0.05** → Data stasioner ✅
- **p-value > 0.05** → Data TIDAK stasioner ❌

In [ ]:
# Test stationarity
def test_stationarity(data, nama):
    result = adfuller(data.dropna())
    p_val = result[1]
    status = '✅ STASIONER' if p_val < 0.05 else '❌ TIDAK STASIONER'
    print(f'  {nama:30s} p-value = {p_val:.6f} → {status}')
    return p_val

print('📊 Augmented Dickey-Fuller Test')
print('=' * 70)
test_stationarity(df['Passengers'], 'Data asli')
test_stationarity(df['Passengers'].diff(), 'Setelah differencing 1x')
test_stationarity(df['Passengers'].diff().diff(), 'Setelah differencing 2x')
print('=' * 70)

# Visualisasi
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(df['Passengers'], color='#2196F3')
axes[0].set_title('Data Asli\n(Tidak Stasioner)', fontsize=11, fontweight='bold')

axes[1].plot(df['Passengers'].diff(), color='#4CAF50')
axes[1].set_title('Differencing 1x\n(Hampir Stasioner)', fontsize=11, fontweight='bold')

axes[2].plot(df['Passengers'].diff().diff(), color='#FF6F00')
axes[2].set_title('Differencing 2x\n(Stasioner)', fontsize=11, fontweight='bold')

for ax in axes:
    ax.set_xlabel('Tahun', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Efek Differencing pada Data', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## 5️⃣ Forecasting dengan SARIMA

**SARIMA** = Seasonal ARIMA — model yang bisa menangkap **trend, musiman, dan autokorelasi** sekaligus.

SARIMA punya parameter:
- **(p, d, q)** = komponen non-seasonal
- **(P, D, Q, m)** = komponen seasonal (m = panjang musim, 12 untuk data bulanan)

Kita tidak perlu menebak parameter ini — `auto_arima` akan **mencari otomatis**!

### Train/Test Split

Untuk time series, kita **tidak bisa mengacak** data. Data diurutkan berdasarkan waktu, jadi kita ambil data terakhir sebagai test.

In [ ]:
# Split: 12 bulan terakhir sebagai test
n_test = 12
train = df['Passengers'][:-n_test]
test = df['Passengers'][-n_test:]

print(f'Data latihan: {len(train)} bulan ({train.index[0].strftime("%b %Y")} — {train.index[-1].strftime("%b %Y")})')
print(f'Data ujian:   {len(test)} bulan ({test.index[0].strftime("%b %Y")} — {test.index[-1].strftime("%b %Y")})')

# Visualisasi split
plt.figure(figsize=(14, 5))
plt.plot(train, color='#2196F3', label='Data Latihan', linewidth=1.5)
plt.plot(test, color='#FF6F00', label='Data Ujian (12 bulan terakhir)', linewidth=2)
plt.axvline(x=test.index[0], color='red', linestyle='--', alpha=0.5)
plt.xlabel('Tahun', fontsize=11)
plt.ylabel('Penumpang', fontsize=11)
plt.title('Train/Test Split untuk Time Series', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Auto ARIMA — otomatis cari parameter terbaik
print('🔍 Mencari model SARIMA terbaik (ini mungkin butuh waktu ~30 detik)...\n')

model = pm.auto_arima(
    train,
    seasonal=True,     # Aktifkan komponen musiman!
    m=12,              # Panjang musim = 12 bulan
    d=1,               # Differencing 1x (dari ADF test)
    trace=True,        # Tampilkan proses pencarian
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
)

print(f'\n🏆 Model terbaik: {model.order} x {model.seasonal_order}')
print(f'   AIC = {model.aic():.1f} (makin kecil makin baik)')

In [ ]:
# Prediksi 12 bulan ke depan
forecast, conf_int = model.predict(n_periods=n_test, return_conf_int=True)
forecast_index = test.index

# Visualisasi prediksi vs kenyataan
plt.figure(figsize=(14, 6))
plt.plot(train[-36:], color='#2196F3', label='Data Latihan (3 tahun terakhir)', linewidth=1.5)
plt.plot(test, color='#FF6F00', label='Data Asli (aktual)', linewidth=2, marker='o', markersize=5)
plt.plot(forecast_index, forecast, color='#4CAF50', label='Prediksi SARIMA', linewidth=2,
         linestyle='--', marker='s', markersize=5)
plt.fill_between(forecast_index, conf_int[:, 0], conf_int[:, 1],
                 alpha=0.2, color='#4CAF50', label='Interval Kepercayaan 95%')

plt.xlabel('Tahun', fontsize=12)
plt.ylabel('Jumlah Penumpang', fontsize=12)
plt.title('Prediksi SARIMA vs Data Aktual', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('🟠 Titik oranye = data sebenarnya')
print('🟢 Titik hijau = prediksi model')
print('🟢 Area hijau muda = rentang prediksi (model 95% yakin di area ini)')

### Evaluasi Prediksi

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
mape = np.mean(np.abs((test - forecast) / test)) * 100

print('📊 Evaluasi Prediksi')
print('=' * 45)
print(f'  MAE  (Rata-rata Kesalahan):  {mae:.1f} penumpang')
print(f'  RMSE (Root Mean Sq Error):   {rmse:.1f} penumpang')
print(f'  MAPE (Kesalahan Persen):     {mape:.1f}%')
print('=' * 45)

if mape < 10:
    print('\n✅ MAPE < 10% — Prediksi sangat bagus!')
elif mape < 20:
    print('\n✅ MAPE < 20% — Prediksi cukup bagus.')
else:
    print('\n⚠️ MAPE > 20% — Prediksi kurang akurat, model perlu ditingkatkan.')

# Tabel perbandingan
comparison = pd.DataFrame({
    'Bulan': test.index.strftime('%b %Y'),
    'Aktual': test.values,
    'Prediksi': forecast.round(0).astype(int),
    'Selisih': (test.values - forecast).round(0).astype(int)
})
print('\nPerbandingan per bulan:')
comparison

---

## 6️⃣ Forecasting ke Masa Depan

Sekarang kita latih ulang model dengan **semua data**, lalu prediksi 24 bulan ke depan.

In [ ]:
# Latih ulang dengan semua data
model_full = pm.auto_arima(
    df['Passengers'],
    seasonal=True, m=12, d=1,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
)

# Prediksi 24 bulan ke depan
n_future = 24
future_forecast, future_ci = model_full.predict(n_periods=n_future, return_conf_int=True)
future_index = pd.date_range(start=df.index[-1] + pd.DateOffset(months=1), periods=n_future, freq='MS')

# Visualisasi
plt.figure(figsize=(14, 6))
plt.plot(df['Passengers'], color='#2196F3', label='Data Historis', linewidth=1.5)
plt.plot(future_index, future_forecast, color='#4CAF50', linewidth=2,
         linestyle='--', label='Prediksi 2 Tahun ke Depan')
plt.fill_between(future_index, future_ci[:, 0], future_ci[:, 1],
                 alpha=0.2, color='#4CAF50')

plt.xlabel('Tahun', fontsize=12)
plt.ylabel('Jumlah Penumpang', fontsize=12)
plt.title('Prediksi Penumpang 2 Tahun ke Depan', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print('💡 Perhatikan: prediksi menangkap TREND naik dan POLA MUSIMAN!')
print('   Area hijau muda = ketidakpastian yang makin besar seiring waktu.')
print('   Ini masuk akal: prediksi jauh ke depan memang LEBIH TIDAK PASTI.')

---

## 📝 Ringkasan: Apa yang Kita Pelajari Hari Ini?

### Konsep Utama

1. **Time Series** = data berurutan berdasarkan waktu, dengan 3 komponen: trend, seasonality, noise
2. **Stationarity** penting — kebanyakan model membutuhkan data yang stabil. Differencing membantu menstabilkan data
3. **SARIMA** = model yang menangkap trend, musiman, dan autokorelasi → cocok untuk data dengan pola musiman
4. `auto_arima` secara otomatis mencari parameter SARIMA terbaik
5. Prediksi makin jauh ke depan → **interval kepercayaan makin lebar** (makin tidak pasti)

### MAPE (Mean Absolute Percentage Error)
| MAPE | Kualitas |
|------|----------|
| < 10% | Sangat bagus |
| 10-20% | Bagus |
| 20-50% | Cukup |
| > 50% | Buruk |

### Kapan Menggunakan Time Series Analysis?

✅ Data **berurutan berdasarkan waktu** (harian, bulanan, tahunan)
✅ Ada **pola yang berulang** (musiman)
✅ Ingin **memprediksi nilai di masa depan**

Contoh: prediksi penjualan, cuaca, harga saham, jumlah pengunjung, konsumsi listrik

---

## 🎉 Selamat! Modul 1 Selesai!

Kamu telah mempelajari **9 topik** Machine Learning Fundamentals:

| # | Topik | Jenis |
|---|-------|-------|
| 1 | Linear Regression | Supervised - Regresi |
| 2 | Logistic Regression | Supervised - Klasifikasi |
| 3 | Decision Tree | Supervised - Klasifikasi |
| 4 | K-Nearest Neighbors | Supervised - Klasifikasi |
| 5 | Support Vector Machines | Supervised - Klasifikasi |
| 6 | Ensemble Methods | Supervised - Ensemble |
| 7 | XGBoost | Supervised - Ensemble |
| 8 | Clustering | **Unsupervised** |
| 9 | Time Series | Forecasting |

### 🔜 Modul Selanjutnya: Deep Learning Fundamentals
Di modul berikutnya, kita akan memasuki dunia **Neural Networks** — model yang terinspirasi dari cara kerja otak manusia!